# **Project Summary -**

The central business problem is:

How can FedEx Logistics analyze shipment patterns, delivery performance, and shipment characteristics to identify delays, operational issues, and opportunities to improve logistics efficiency and reduce costs?


**Shipment management**

Which shipment modes are most frequently used?

Which countries/regions have the highest number of shipments?

Are shipments managed by particular teams more likely to be delivered late?

Does shipment mode affect delivery performance?

**Delivery performance**

What percentage of shipments are delivered on time vs delayed?

Which shipment modes have the highest delays?

Which countries or regions experience more delivery delays?

Are there differences in delivery performance based on shipment mode?

**Vendor / INCO terms**
Does the INCO Term affect vendor delivery performance?

Which INCO terms are associated with more delays?

Does vendor type/location influence delivery performance?

**Product/shipment characteristics**
Does shipment weight have any relationship with insurance?

Are heavier shipments more likely to have higher insurance costs?

Do certain product categories or dosage forms experience more delays?


# **Problem Statement**


**FedEx Logistics manages a complex global supply chain dealing with shipments across various regions, countries, and industries.**

**Effective management of these processes ensures timely delivery, minimize freight costs, and improves customer satisfaction.**

**Rise of ecommerce and global distribution, companies like FedEx must continuously optimize their logistic operations to maintain a competitive edge.**

**The dataset is designed to provide insights into how shipments are managed identify bottle necks or delays, and ensure cost-effectiveness.**

**Streamline supply chain operation, Improving delivery timeline, Reducing costs for both the company and customers.**

## ***1. Know Your Data***

### Import Libraries

In [ ]:
# Data Manipulation
import pandas as pd

# Numerical Computation
import numpy as np

# Visualizations

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# To ignore warnings
import warnings
warnings.filterwarnings('ignore')

### Dataset Loading

In [ ]:
# File Path
file_path = "/content/SCMS_Delivery_History_Dataset.csv"
# Read CSV file
df = pd.read_csv(file_path)

### Dataset First View

In [ ]:
# View first five rows
df.head()

In [ ]:
# View last five rows
df.tail()

### Dataset Rows & Columns count

In [ ]:
# Dataset Rows & Columns count
print(f"Number of Rows: {df.shape[0]}")
print(f"Number of Columns: {df.shape[1]}")

### Dataset Information

In [ ]:
# Dataset Info
df.info()

#### Duplicate Values

In [ ]:
# Dataset Duplicate Value Count
df.duplicated().sum()

No dulicate records were found in the dataset.

#### Missing Values/Null Values

In [ ]:
# Create table for Missing Values, Missing Percentage
missing_table = pd.DataFrame({
    'Missing Values': df.isnull().sum(),
    'Missing Percentage': df.isnull().sum() / len(df) * 100
})
# Select values where Missing Values is grater than zero then sort the values in descending order
missing_table = missing_table[
    missing_table['Missing Values'] > 0
].sort_values(
    'Missing Percentage',
    ascending=False
)

missing_table


In [ ]:
import missingno as msno

# Plot the distribution of missing values
msno.bar(df)


### What did you know about your dataset?

Size of dataset 10324 rows and 33 columns.
data type present are float, int, object

memory usage 2.6+ MB

no duplicate values

but null values present for
3 attribute

Shipment Mode 360 count, Dosage 1736, Line Item Insurance (USD) 287

## ***2. Understanding Your Variables***

In [ ]:
# Dataset Columns
df.columns


In [ ]:
# Rename the columns
df.rename(columns={'Project Code': 'Project_Code',
                  'PQ #':'PQ#',
                  'PO / SO #':'PO/SO#',
                  'ASN/DN #':'ASN/DN#',
                  'Managed By':'Managed_By',
                  'Fulfill Via':'Fulfill_Via',
                  'Vendor INCO Term':'Vendor_INCO_Term',
                  'Shipment Mode':'Shipment_Mode',
                  'PQ First Sent to Client Date':'PQ_First_Sent_To_Client_Date',
                  'PO Sent to Vendor Date':'PO_Sent_To_Vendor_Date',
                  'Scheduled Delivery Date':'Scheduled_Delivery_Date',
                  'Delivered to Client Date':'Delivered_To_Client_Date',
                  'Delivery Recorded Date':'Delivery_Recorded_Date',
                  'Product Group':'Product_Group',
                  'Sub Classification':'Sub_Classification',
                  'Item Description':'Item_Description',
                  'Molecule/Test Type':'Molecule/Test_Type',
                  'Dosage Form':'Dosage_Form',
                  'Unit of Measure (Per Pack)':'Unit_Of_Measure_(Per_Pack)',
                  'Line Item Quantity':'Line_Item_Quantity',
                  'Line Item Value':'Line_Item_Value',
                  'Pack Price':'Pack_Price',
                  'Unit Price':'Unit_Price',
                  'Manufacturing Site':'Manufacturing_Site',
                  'First Line Designation':'First_Line_Designation',
                  'Weight (Kilograms)':'Weight_(kg)',
                  'Freight Cost (USD)':'Freight_Cost_(USD)',
                  'Line Item Insurance (USD)':'Line_Item_Insurance_(USD)'}, inplace=True)

In [ ]:
# Convert DateTime
date_columns = ['PQ_First_Sent_To_Client_Date', 'PO_Sent_To_Vendor_Date',
                'Scheduled_Delivery_Date','Delivered_To_Client_Date','Delivery_Recorded_Date']

for col in date_columns:
    df[col] = pd.to_datetime(df[col],errors='coerce')
df[date_columns].dtypes

In [ ]:
# Check for null value for 'PQ_First_Sent_To_Client_Date', 'PO_Sent_To_Vendor_Date','Scheduled_Delivery_Date','Delivered_To_Client_Date','Delivery_Recorded_Date'
df[date_columns].isnull().sum()

In [ ]:
# Dataset Describe
# Condition only consider float64 and int64
condition = (df.dtypes == 'float64') | (df.dtypes == 'int64')
df.loc[:, condition].describe()

In [ ]:
df.describe(include='object')

### Variables Description

out off 33 only 7 columns are int or float type all other are object include Weight_(kg) and Freight_Cost_(USD)

We convert this four columns PQ_First_Sent_To_Client_Date, PO_Sent_To_Vendor_Date, Scheduled_Delivery_Date, Delivered_To_Client_Date,Delivery_Recorded_Date in datetime

We have null values for PQ_First_Sent_To_Client_Date and PO_Sent_To_Vendor_Date


### Check Unique Values for each variable.

In [ ]:
# Check Unique Values for each variable.
categorical_columns = df.select_dtypes(include='object').columns

for col in categorical_columns:
  print("\n" + "="*50)
  print(col)
  print("Number of unique values:", df[col].nunique())
  print(df[col].dropna().unique()[:20])

## 3. ***Data Wrangling***

### Data Wrangling Code

In [ ]:
# Clean Extra Spaces
for col in df.select_dtypes(include='object').columns:
  df[col] = df[col].str.strip()

In [ ]:
# Replace Côte d'Ivoire with Ivory Coast
df['Country']=df['Country'].replace({"Côte d'Ivoire":"Ivory Coast"})
df['Country'].head()

In [ ]:
# Create Delivery Delay
'''
Delivery Delay = Delivered_To_Client_Date - Scheduled_Delivery_Date
'''
df['Delivery_Delay'] = (df['Delivered_To_Client_Date'] - df['Scheduled_Delivery_Date']).dt.days
df['Delivery_Delay'].describe()


In [ ]:
# Check
condition = ['Scheduled_Delivery_Date','Delivered_To_Client_Date','Delivery_Delay']
df[condition].head()

In [ ]:
# Create Delivery Status
df['Delivery_Status'] = np.where(df['Delivery_Delay'] > 0, 'Late', 'On Time')
df['Delivery_Status'].value_counts()

In [ ]:
df['Delivery_Status'].value_counts(normalize=True)*100

In [ ]:
# Clean Weight Column
import re

# Create ID -> Weight mapping
id_weight_map = df.set_index('ID')['Weight_(kg)'].to_dict()

def clean_weight(value):
    value = str(value)

    # If already numeric, keep it
    if value.isdigit():
        return int(value)

    # Extract ID from strings like:
    # "See DN-4282 (ID#:83919)"
    match = re.search(r'ID#:(\d+)', value)

    if match:
        lookup_id = int(match.group(1))
        return id_weight_map.get(lookup_id)

    return None

df['Weight_(kg)_new'] = df['Weight_(kg)'].apply(clean_weight)
df[['Weight_(kg)_new','Weight_(kg)']]

In [ ]:
# covert weight in int or float
df['Weight_(kg)_new'] = pd.to_numeric(df['Weight_(kg)_new'],errors='coerce')
df[['Weight_(kg)','Weight_(kg)_new']].head(10)

In [ ]:
df['Weight_(kg)_new'].describe()

In [ ]:
df['Freight_Cost_(USD)'].unique()

In [ ]:
# Clean Freight Cost
def is_float(val):
    try:
        float(val)
        return True
    except ValueError:
        return False
# Create ID -> freight Cost mapping
id_freight_cost_map = df.set_index('ID')['Freight_Cost_(USD)'].to_dict()

def clean_freight_cost(value):
    value = str(value)

    # If already numeric, keep it
    if is_float(value):
        return float(value)

    # Extract ID from strings like:
    # "See DN-4282 (ID#:83919)"
    match = re.search(r'ID#:(\d+)', value)

    if match:
        lookup_id = int(match.group(1))
        return id_freight_cost_map.get(lookup_id)

    return None

df['Freight_Cost_(USD)_new'] = df['Freight_Cost_(USD)'].apply(clean_freight_cost)
df[['Freight_Cost_(USD)_new','Freight_Cost_(USD)']]

In [ ]:
# Convert into numeric
df['Freight_Cost_(USD)_new'] = pd.to_numeric(df['Freight_Cost_(USD)_new'], errors='coerce')
df['Freight_Cost_(USD)_new'].describe()

In [ ]:
# Check Numeric Columns
df.select_dtypes(include = np.number).columns

In [ ]:
df.select_dtypes(include = np.number).describe().T

In [ ]:
# Check Categorical Variables
df['Shipment_Mode'].value_counts(dropna=False)

In [ ]:
df['Country'].value_counts().head(20)

In [ ]:
# Vendor_INCO_Term
df['Vendor_INCO_Term'].value_counts(dropna=False)

In [ ]:
# Product Group
df['Product_Group'].value_counts()

In [ ]:
# Missing Value Treatment
df['Shipment_Mode'] = df['Shipment_Mode'].fillna('Unknown')
df['Dosage'] = df['Dosage'].fillna('Unknown')

In [ ]:
# Final Data Check
print("Dataset shape:",df.shape)
print("Duplicate rows:",df.duplicated().sum())

In [ ]:
df.info()

In [ ]:
df.head()

In [ ]:
df.isnull().sum().sort_values(ascending=False).head(15)

In [ ]:
# Save the Cleaned Dataset
df.to_csv('SCMS_Deilvery_History_Cleaned.csv', index = False)

### What all manipulations have you done and insights you found?

I performed the following data preprocessing and manipulation steps:

1. **Loaded the dataset using Pandas** and checked its shape and structure.

2. **Checked duplicate** records and **found no duplicate rows**.

3. **Removed extra spaces** from categorical/text columns to maintain consistency.

4. **Checked missing values** and calculated both missing counts and percentages.

5. **Converted date** columns **into datetime** format.

6. **Created a new Delivery Delay** column by calculating the difference between the actual delivery date and scheduled delivery date.

7. **Created a Delivery Status column**:

  - Delay > 0 → Delayed

  - Delay ≤ 0 → On Time

8. **Converted Weight and Freight Cost into numerical columns** because the original columns contained both numeric and text values.

9. **Handled categorical missing values** such as Shipment Mode and Dosage by replacing missing values with **"Unknown"**.


## ***4. Data Vizualization, Storytelling & Experimenting with charts : Understand the relationships between variables***

#### Chart - 1 How are shipments distributed across shipment modes ?

In [ ]:
shipment_mode_count = df['Shipment_Mode'].value_counts()

shipment_mode_count.sort_values(ascending=False)

In [ ]:
plt.figure(figsize=(8,5))

sns.countplot(data = df,
              x = 'Shipment_Mode',
              order = df['Shipment_Mode'].value_counts(ascending=False).index)

plt.title('Distribution of Shipment Modes')
plt.xlabel('Shipment Mode')
plt.ylabel('Count of Shipments')
plt.show()

##### 1. Why did you pick the specific chart?

I selected a bar chart because shipment mode is a catrgorical variable, and a bar chart makes it easy to compare the number of shipments handled by each mode.

##### 2. What is/are the insight(s) found from the chart?

- **Air** is the **most frequently used shipment mode** with 6113 shipments.
- Truck has 2380 shipments.
- Air Charter has 650 shipments.
- Ocean has 371 shipments.
- 360 shipments have an unknown shipment mode.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes. This insight can help the company understand its dependency on different transportation modes and improve transportation planning and resource allocation.

A high dependency on **Air transportation may increase to transportation-cost fluctuations.** Also, the presence of **360 unknown shipment-mode recordes indicates a data-quality issue**, which can affect operational analysis.


#### Chart - 2 What percentage of shipments are delayed ?

In [ ]:
# What percentage of shipments are delayed ?
delivery_status = df['Delivery_Status'].value_counts()
delivery_status

In [ ]:
delivery_percentage = (
    df['Delivery_Status']
    .value_counts(normalize = True)
    .mul(100)
    .round(2)
    )
delivery_percentage

In [ ]:
plt.figure(figsize = (8,5))
y = df['Delivery_Status'].value_counts()
labels = ['On Time', 'Delayed']
plt.pie(y, labels = labels)
plt.show()

##### 1. Why did you pick the specific chart?

I selected a pie chart because I want to show the proportion of total shipments that were delivered on time versus delayed.

##### 2. What is/are the insight(s) found from the chart?

- 9138 shipments (**88.51%**) were **on time**.

- 1186 shipments(**11.49%**) were **delayed**.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes. The company can use this information to monitor delivery performance and investigate the causes of delayed shipments.

The **11.49% delayed** shipments represent a potential operational problem. Delays can result in customer dissatisfaction, additional logistics costs, and possible loss of business if the issue is not addressed.

#### Chart - 3 Does shipment mode affect delivery performance ?

In [ ]:
mode_df = df[df['Shipment_Mode'] != 'Unknown']

In [ ]:
mode_delivery = pd.crosstab(
    mode_df['Shipment_Mode'],
    mode_df['Delivery_Status'],
    normalize='index'
) * 100
mode_delivery.round(2)

In [ ]:
plt.figure(figsize = (10,6))

sns.countplot(
    data = mode_df,
    x = 'Shipment_Mode',
    hue = 'Delivery_Status'
)

plt.title('Shipment Mode vs Delivery Status')
plt.xlabel('Shipment Mode')
plt.ylabel('Count')
plt.show()

##### 1. Why did you pick the specific chart?

I selected a stacked bar chart because it allowed me to compare both On time and Delayed shipments across different shipment modes in one visualization.

##### 2. What is/are the insight(s) found from the chart?

In this dataset, the observed delayed-shipment percentage is:

**Air : 9.60%**

**Air Charter : 11.54%**

**Truck : 16.08%**

**Ocean : 17.52%**

So delivery performance varies across shipment modes.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes, This can help logistic teams investigate why certain transportation modes experience more delays and improve shipment planning.

The relatively higher delay rates for **Ocean and Truck** may increase delivery-related costs and negativly affect customer satisfaction.

#### Chart - 4 What is the average delivery delay for each shipment mode ?

In [ ]:
mode_delay = (
    mode_df.groupby('Shipment_Mode')['Delivery_Delay']
    .agg(['count', 'mean', 'median'])
    .sort_values('mean', ascending = False)
)

mode_delay.round(2)

In [ ]:
plt.figure(figsize = (12, 6))

sns.barplot(
    data = mode_delay.reset_index(),
    x = 'Shipment_Mode',
    y = 'mean'
)

plt.title('Average Delivery Delay by Shipment Mode')

##### 1. Why did you pick the specific chart?

I selected a bar chart because I want to compare Average Delivery Delay by Shipment Mode.

##### 2. What is/are the insight(s) found from the chart?

- Ocean average delivery delay 5.87.
- Air average delivery delay -3.76.
- Truck average delivery delay -9.92.
- Air Charter average delivery delay -19.04.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes. It shows how shipment mode effect on Delivery Delay.

This analysis compares delivery performance across shipment modes. **Ocean has the highest average delay** of 5.87 days, whereas Air Charter has the **lowest average dela**y of −19.04 days. Negative delay means the shipment was delivered before its scheduled date. Therefore, Ocean transportation appears to require further investigation from a delivery-performance perspective.

#### Chart - 5 Which countries have higher delivery delays ?

In [ ]:

country_delay = (
    df.groupby('Country')['Delivery_Delay'].agg(['count','mean'])
)

country_delay = country_delay[country_delay['count']>=20].sort_values(
    'mean',
    ascending = False
)
country_delay.head(5)

In [ ]:
plt.figure(figsize=(12,6))

sns.barplot(
    data = country_delay.head(10).reset_index(),
    x = 'mean',
    y = 'Country'
)

plt.title('Top 10 Countries by Average Delivery Delay')
plt.xlabel('Average Delivery Delay (Days)')
plt.ylabel('Country')

plt.show()

##### 1. Why did you pick the specific chart?

I selected a bar chart becasue I want to compare averge delivery delays across different countries.

##### 2. What is/are the insight(s) found from the chart?

Among countries with **at least 20 shipments, Congo, DRC had an average delay of approximately 11.24 days**, which was considerably higher than several other countries.


##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes. Country-level analysis can help identify locations where logistics operations may require additional attention.

High delays in particular countries may indicate transportation, customs, infrastructure, or supply-chain challenges. Continued delays can affect customer srvice and operational costs.

#### Chart - 6 Which countries have the highest shipment volumnes ?

In [ ]:

country_volume = (
    df['Country']
    .value_counts()
    .head(10)
)

country_volume

In [ ]:
plt.figure(figsize = (12, 6))

sns.barplot(
    x = country_volume.values,
    y = country_volume.index
)

plt.title('Top 10 Countries by Number of Shipments')
plt.xlabel('Number of Shipments')
plt.ylabel('Country')

plt.show()

##### 1. Why did you pick the specific chart?

I pick bar chart to compare country and Number of shipments.

##### 2. What is/are the insight(s) found from the chart?

**South Africa has the highest number of shipments** with 1,406, followed by Nigeria with 1,194 and Ivory Coast with 1,083.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

The insight helps us identify high-volume countries where logistics resources and operational attention should be prioritized. However, shipment volume alone does not indicate delivery performance, so I would combine this analysis with country-wise delivery delay to make business decisions

#### Chart - 7 Does INCO Term relate to delivery performance ?

In [ ]:
inco_analysis = (
    df.groupby('Vendor_INCO_Term')['Delivery_Delay']
    .agg(['count', 'mean', 'median'])
    .sort_values('mean', ascending = False)
)

inco_analysis.round(2)

In [ ]:
plt.figure(figsize = (10,6))

sns.barplot(
    data = inco_analysis.reset_index(),
    x = 'Vendor_INCO_Term',
    y = 'mean'
)

plt.title('Average Delivery Delay by Vendor INCO Term')
plt.xlabel('Vendor INCO Term')
plt.ylabel('Average Delivery Delay (Days)')

plt.xticks(rotation = 45)
plt.show()

##### 1. Why did you pick the specific chart?

A bar chart makes it easy to compare average delivery delay across different Vendor INCO Trems.

##### 2. What is/are the insight(s) found from the chart?

The average delivery delay differs between INCO terms. For example:

- DDP: -12 days
- DDU : -2 days
- FCA : half days
- N/A - From RDC : -8 days

A negative value means the shipment was delivered before the scheduled delivery date.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes, Understnading delivery performance under different commercial/shipping terms can help the company evaluate logistics processes and scheduling.

Some categories have very small sizes, such as DDU with only 15 records. Therefore, making a strong business decision based only on those categories could be misleading.

#### Chart - 8  Which preduct groups have the highest shipment volume ?

In [ ]:
product_count = df['Product_Group'].value_counts()
product_count

In [ ]:
plt.figure(figsize=(8,5))

sns.countplot(
    data = df,
    x = 'Product_Group',
    order = product_count.index
)

plt.title('Shipment Distribution by Product Group')
plt.xlabel('Product Group')
plt.ylabel('Number of Shipments')

plt.show()

##### 1. Why did you pick the specific chart?

I select bar chart to see shipment Distribution by Product Group. It is easy to compare numerical data with categorical data.

##### 2. What is/are the insight(s) found from the chart?

**ARV is the dominant product group**, accounting for approximately 82.8% of all 10,324 shipment records. HRDT accounts for approximately 16.7%, while the **remaining three groups together account for less than 0.5%.**

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Help the company prioritize inventory, logistics, and operational resources for high-volume products.
However, the very small sample sizes of ANTM, ACT, and MRDT mean that we should be careful when making business decisions based on their delivery performance.

#### Chart - 9

In [ ]:
# Does product group have different delivery performance ?
product_delay = (
    df.groupby('Product_Group')['Delivery_Delay']
    .agg(['count','mean','median'])
    .sort_values('mean', ascending = False)
)
product_delay.round(2)

In [ ]:
plt.figure(figsize = (9, 5))

sns.barplot(
    data = product_delay.reset_index(),
    x = 'Product_Group',
    y = 'mean'
)

plt.title('Average Delivery Delay by Product Group')
plt.xlabel('Product_Group')
plt.ylabel('Average Delivery Delay (Days)')

plt.show()

##### 1. Why did you pick the specific chart?

A bar chart is appropriate because Product Group is categorical, and I want to compare the number of shipments belonging to each product group.

##### 2. What is/are the insight(s) found from the chart?

- ARV:8550
- HRDT : 1728
- ANTM : 22
- ACT : 16
- MRDT : 8

ARV represents the vast majority of records.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes, This can help the company understand which product groups generate the highest shipment volume and therefor require greater operational attention.

The dataset is **highly concentrated in ARV. Very small groups such as MRDT, ACT, and ANTM have too few observation**s to support reliable conclusions about their delivery performance.

#### Chart - 10 Are heavier shipments associated with higher insurance values ?

In [ ]:
weight_insurance = df[
    ['Weight_(kg)_new','Line_Item_Insurance_(USD)']
].dropna()

In [ ]:
correlation = weight_insurance.corr().loc[
    'Weight_(kg)_new',
    'Line_Item_Insurance_(USD)'
]

print(f'Correation:{round(correlation,3)}')

In [ ]:
plt.figure(figsize = (10,6))

sns.scatterplot(
    data = weight_insurance,
    x = 'Weight_(kg)_new',
    y = 'Line_Item_Insurance_(USD)'
)

plt.title("Wright vs Insurance")
plt.xlabel("Weight (kg)")
plt.ylabel("Insurance (USD)")

plt.show()

##### 1. Why did you pick the specific chart?

I selected a scatter plot because both Weight and Insurance are numerical variables. A scatter plot helps identify  whether there is a relationship between them.

##### 2. What is/are the insight(s) found from the chart?

The correlation between **weight and insurance** is approximately 0.53, indicating a moderate **positive relationship**.

In general, **heavier shipments tend to be associated with higher insurance values**.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes. This information can help the company understand how shipment characteristics relate to insurance requirements and costs.

Higher-weight shipments may involve higher insurance costs. However, correlation does not prove that weight itself causes higher insurance costs.

#### Chart - 11 Does freight cost have a relationship with delivery delay ?

In [ ]:

freight_delay = df[
    ['Freight_Cost_(USD)_new', 'Delivery_Delay']
].dropna()

freight_delay.corr()

In [ ]:
plt.figure(figsize = (10,6))

sns.scatterplot(
    data = freight_delay,
    x = 'Freight_Cost_(USD)_new',
    y = 'Delivery_Delay'
)

plt.title('Freight Cost vs Delivery Delay')
plt.xlabel('Freight Cost (USD)')
plt.ylabel('Delivery Delay (Days)')

plt.show()

##### 1. Why did you pick the specific chart?

A scatter plot is suitable because both Freight Cost and Delivery Delay are numerical variables, and I want to examine their relationship.

##### 2. What is/are the insight(s) found from the chart?

The correlation is approximately -0.025, which is **very close to zero**.

This suggests that there is **little linear relationship between freight cost and delivery delay** in this dataset.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes. It suggests that simply increasing or decreasing freight cost may not directly solve the delivery-delay problem. The company may need to investigate other operational factors.

If management assumes that higher freight spending automatically reduces delays, it could result in unnecessary logistics expenditure without significantly improving delivery performance.

#### Chart - 12 How does fulfillment method relate to delivery perfromance ?

In [ ]:
fulfill_analysis = (
    df.groupby('Fulfill_Via')['Delivery_Delay']
    .agg(['count', 'mean'])
    .sort_values('mean', ascending = False)
)

fulfill_analysis.round(2)

In [ ]:
plt.figure(figsize=(8,5))

sns.barplot(
    data = df,
    x = 'Fulfill_Via',
    y = 'Delivery_Delay'
)

plt.title('Average Delivery Delay by Fulfillment Method')
plt.xlabel('Fulfillment Method')
plt.ylabel('Average Delivery Delay (Days)')

plt.show()

##### 1. Why did you pick the specific chart?

A bar chart allows us to compare the average delivery delay between different fulfillent methods.

##### 2. What is/are the insight(s) found from the chart?

Fulfillment Method - Average Delay

**Direct Drop - -3.54 days**

**From RDC - -8.28 days**


Both averages are negative, meaning shipments were deliered earlier than their scheduled dates on average.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes. This can help the company compare fullfillment approaches and understand their delivery performance.

The analysis alone does not show that one fulfillment method causes better performance. Other factors such as country, shipment mode, or product group may also influence delivery time.

#### Chart - 13

In [ ]:
fig = px.bar(
    df['Shipment_Mode'].value_counts().reset_index(),
    x = 'Shipment_Mode',
    y = 'count',
    title='Shipment Mode Distribution'

)
fig.show()

In [ ]:
fig = px.box(
    df[df['Shipment_Mode']!='Unknown'],
    x = 'Shipment_Mode',
    y = 'Delivery_Delay',
    color = 'Shipment_Mode',
    title = 'Shipment Mode vs Delivery Delay'
)
fig.show()

##### 1. Why did you pick the specific chart?

Box plot is the best way for Anomaly detection and statistical analysis.

##### 2. What is/are the insight(s) found from the chart?

Ocean shipments show a greater tendency toward positive delivery delays, while Truck and Air Charter generally have negative delivery delays, meaning they tend to be delivered earlier than scheduled.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

This insight can help the logistics team identify shipment modes that require closer monitoring. In particular, Ocean shipments show greater delivery-delay concerns, so the company can investigate the reasons for these delays and improve scheduling or transportation planning

#### Chart - 14 - Correlation Heatmap

In [ ]:
# Correlation Heatmap visualization code
# Correlation Heatmap

numeric_columns = [
    'Unit_Of_Measure_(Per_Pack)',
    'Line_Item_Quantity',
    'Line_Item_Value',
    'Pack_Price',
    'Unit_Price',
    'Weight_(kg)_new',
    'Freight_Cost_(USD)_new',
    'Line_Item_Insurance_(USD)',
    'Delivery_Delay'
]

correlation_matrix = df[numeric_columns].corr()

mask = np.triu(np.ones_like(correlation_matrix, dtype = bool))

plt.figure(figsize = (12, 8))

sns.heatmap(
    correlation_matrix,
    mask = mask,
    annot = True,
    fmt = '.2f',
    cmap = 'coolwarm'
)
plt.title('Correlation Matrix of Numerical Variables')
plt.show()

##### 1. Why did you pick the specific chart?

I selected a correlation heatmap because the dataset contains several numerical variables. A heatmap allows me to quickly identify positive and negative relationships between these variables.

##### 2. What is/are the insight(s) found from the chart?

- **Weight and insurance** show a moderate **positive relationship**.
- **Freight cost and delivery delay** have almost **no linear relationship**.
- The heatmap helps identify variables that may be useful for futher analysis or modeling.

#### Chart - 15 - Pair Plot

In [ ]:
# Pair Plot visualization code
pair_df = df[[
    'Line_Item_Quantity',
    'Line_Item_Value',
    'Pack_Price',
    'Unit_Price',
    'Weight_(kg)_new',
    'Freight_Cost_(USD)_new',
    'Line_Item_Insurance_(USD)',
    'Delivery_Delay'
]].dropna()

# Create the pair plot

sns.pairplot(pair_df)
plt.show()

##### 1. Why did you pick the specific chart?

For Multivariate analysis in one chart we can find out all positive or negative correlation or zero correlation.

##### 2. What is/are the insight(s) found from the chart?

Weight and insurance show a moderate positive relationship.

Freight cost and delivery delay have almost no linear relationship.

## **5. Solution to Business Objective**

#### What do you suggest the client to achieve Business Objective ?
Explain Briefly.

1. **Focus on improving Ocean shipment performance**

  Ocean shipments showed the highest average delivery delay, approximately 5.87 days, and a relatively high delay rate.

  The company should investigate the reasons behind Ocean shipment delays, such as scheduling, transit time, or operational bottlenecks, and improve planning for Ocean shipments.

  Reducing Ocean delays can improve customer satisfaction and reduce additional logistics costs.

2. **Monitor Truck shipments closely**

  Truck shipments also showed a relatively high delay rate compared with Air shipments.

  The company should monitor Truck transportation routes and delivery schedules and identify locations or routes where delays occur frequently.

  Better route and schedule management can improve delivery reliability.

3. **Prioritize high-volume countries**

  South Africa, Nigeria, and Ivory Coast have the highest shipment volumes in the dataset.

  The company should give greater operational attention and resource planning to high-volume countries.

  Improving efficiency in these locations can have a larger overall impact because they handle a significant proportion of shipments.

4. **Investigate countries with high delivery delays**

  Some countries, particularly Congo, DRC, showed relatively high average delivery delays.

  Perform a country-level investigation to identify the operational factors contributing to delays.

  Targeted improvements can reduce delays without applying the same solution to every country.

5. **Improve data quality**

  The analysis identified missing values, including missing Shipment Mode, Dosage, and Insurance information. Weight and Freight Cost also contained non-numeric values.

  Improve data-entry standards and introduce validation rules in the supply-chain system.

  Higher-quality data will improve reporting, decision-making, and future predictive models.

6. **Use weight information for insurance planning**

  The correlation analysis showed a moderate positive relationship (~0.58) between shipment weight and insurance value.

  The company can use shipment weight as one of the factors when reviewing insurance requirements and costs.

  This may help improve insurance planning and cost estimation.

7. **Do not rely only on freight cost to reduce delays**

  The correlation between Freight Cost and Delivery Delay was approximately −0.025, which is very close to zero.

  The company should not assume that increasing freight expenditure alone will solve delivery delays. Instead, it should investigate operational factors such as shipment mode, country, scheduling, and fulfillment process.

  This can prevent unnecessary spending without a corresponding improvement in delivery performance.

8. **Focus inventory and logistics planning on major product groups**

  ARV accounts for approximately 8,550 shipments, making it the dominant product group.

  The company should prioritize inventory, transportation, and supply-chain planning for high-volume product groups such as ARV and HRDT.

  Improving operations for high-volume products can produce a larger overall business impact.

# **Conclusion**

The analysis of the SCMS Delivery History dataset provided valuable insights into shipment distribution and delivery performance. Approximately 88.51% of shipments were delivered on time, while 11.49% were delayed. Air was the most frequently used shipment mode, whereas Ocean shipments showed relatively higher delivery delays.

The analysis also showed that shipment volumes are concentrated in a few countries and product groups, particularly South Africa, Nigeria, Ivory Coast, and the ARV product group. A moderate positive relationship was observed between shipment weight and insurance value, while freight cost showed almost no linear relationship with delivery delay.

Based on these findings, the client should focus on improving Ocean and Truck shipment performance, monitoring high-delay countries, prioritizing high-volume markets and products, and improving data quality. The company should also investigate the root causes of delays rather than relying only on increasing freight costs.

Overall, the analysis can help the organization make data-driven decisions, improve delivery reliability, optimize logistics operations, reduce unnecessary costs, and ultimately improve customer satisfaction.